# 01 · Quickstart — the CRAF'd Forecast API

**What this teaches:** how to call the live CRAF'd Forecast API end to end — authenticate,
fetch **historical** observations and **probabilistic forecasts**, slice by time / place /
feature / posterior sample, and read **HDI / MAP** uncertainty summaries — at PRIO-GRID and
aggregated administrative levels. The service now serves a **global** forecast — every
land cell worldwide (not just Africa) — and global historical; §2.5 confirms this live.

**Audience:** an analyst or developer consuming the service (the UN CRAF'd data path). Data
subsetting uses the `CrafdApiClient` wrapper; the `sample_idx` and `/analysis` endpoints are
shown as raw HTTP so non-Python consumers can translate them.

> **How to run this notebook.** It calls the **live service**, so it needs an **API key**.
> Copy `.env.example` to `.env` and set `APPWRITE_DATASTORE_API_KEY` (request one from the
> VIEWS team). With no key it reads as documentation; with a key, *Run All* fetches live data.
> **What these numbers are.** Conflict-fatality forecasts — an *input/driver* to food-security analysis, **not** a food-security, IPC, or hunger output. **All values are raw fatality counts**; the `lr_`/`pred_` prefixes are legacy VIEWS naming, **not** a log scale. Every served column is defined in the [data dictionary](../docs/api/data_dictionary.md).


## 1 · Setup & authentication

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.time import date_to_month_id, month_id_to_date, month_id_range

load_dotenv()  # reads APPWRITE_DATASTORE_API_KEY (and optional VIEWS_API_URL) from .env

# Set USE_LOCAL = True to target a locally-running API instead of production.
USE_LOCAL = False
BASE_URL = "http://localhost:8000" if USE_LOCAL else os.environ.get(
    "VIEWS_API_URL", "https://crafdapi.viewsforecasting.org"
)
API_KEY = os.environ.get("APPWRITE_DATASTORE_API_KEY", "").strip().strip('"')
HEADERS = {"X-API-Key": API_KEY}  # for the raw-HTTP examples below

# Generous timeout: the first request after a (re)deploy can take ~1-2 min while
# the server loads and caches the dataset from source; subsequent calls are fast.
client = CrafdApiClient(BASE_URL, API_KEY, timeout=600)

# The forecast run window is month_ids 559-594 (2026-07 … 2029-06); it advances each run.
FORECAST_MONTH = date_to_month_id(2026, 7)   # first forecast month of the current run
HIST_MONTH = date_to_month_id(2024, 12)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING — set it in .env")
# /health needs a valid key; degrade gracefully so a keyless reader still gets past setup.
try:
    print("Health  :", client.health().get("status"))
except Exception as e:
    print(f"Health  : unavailable ({e}) — set APPWRITE_DATASTORE_API_KEY in .env for live cells.")

# Cold-start helper: the FIRST data call on a freshly-(re)deployed server rebuilds the dataset
# (~2-3 min) and can exceed the client timeout; retry once — the server keeps the warmed cache, so
# the retry is fast. Run `scripts/smoke.py` right after a deploy to pre-warm and skip this wait.
def fetch_warm(fn):
    try:
        return fn()
    except requests.exceptions.Timeout:
        print("  server cache was cold — warmed, retrying once…")
        return fn()

Base URL: https://crafdapi.viewsforecasting.org
API key : set


Health  : healthy


In [2]:
# Provenance — which forecast run is live. Record it alongside any number/figure you cite;
# a new run supersedes the previous one, so forecasts are not reproducible across runs.
try:
    prov = client.provenance("forecast")
    print("Forecast run :", prov.get("run_id") or prov.get("name"))
    print("Produced     :", prov.get("created_at"))
    print("Methodology  :", prov.get("methodology_version"))
except Exception as e:
    print(f"Provenance unavailable ({e})")


Forecast run : rusty_bucket_forecasting_20260727_095355
Produced     : 2026-08-14T18:35:54.962+00:00
Methodology  : crafdapi-methodology/3


In [3]:
# --- Preflight 1/2: is there a key at all? ---
# The setup cell above prints MISSING and continues, so without this the first data call
# fails as `401 ... User (role: guests) missing scopes (["buckets.read"])` — an Appwrite
# message about scopes, for what is actually an absent .env.
# `.env.example`'s placeholder counts as "no key": copying the template and forgetting to edit
# it is the common trap, and it fails as `401 ... not authorized` rather than as a missing key.
# `.env.example` ships the literal "your-api-key-here"; copying the template and
# forgetting to edit it is the common trap, and it fails as 401 rather than as a missing key.
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit(
        "No usable API key.\n\n"
        "The live notebooks read APPWRITE_DATASTORE_API_KEY from a .env at the ROOT OF THIS\n"
        "REPOSITORY (views-crafdapi/.env) — not from views-models.\n\n"
        "Fix:\n"
        "    cp .env.example .env\n"
        "    # then EDIT .env — replace `your-api-key-here` with a real key\n"
        "    #   (request one from the VIEWS team)\n\n"
        "Then restart the kernel and re-run from the top. "
        "03_offline_demo.ipynb needs no key at all."
    )

# --- Preflight 2/2: is the month this notebook asks for inside the served run? ---
_prov = client.provenance("forecast")
_served = _prov.get("run_id") or _prov.get("name") or "unknown run"
_probe = client.fetch_subset("country", [FORECAST_MONTH], ["IDN"],
                             data_type="forecast", features=["pred_lr_ged_sb"])
if len(_probe) == 0:
    raise SystemExit(
        f"Preflight failed: month_id {FORECAST_MONTH} "
        f"({month_id_to_date(FORECAST_MONTH)}) returned no rows from the served run "
        f"'{_served}'.\n\n"
        f"The forecast window advances with each delivery, so a month that worked last "
        f"month may now be outside it.\n"
        f"Fix: change FORECAST_MONTH above to a month inside the current run, then re-run "
        f"from the top.\n"
        f"Run produced: {_prov.get('created_at')}"
    )
print(f"Preflight OK — month {FORECAST_MONTH} "
      f"({month_id_to_date(FORECAST_MONTH)}) is served by run '{_served}'.")


Preflight OK — month 559 (2026-07) is served by run 'rusty_bucket_forecasting_20260727_095355'.


## 2 · The VIEWS month-id model

Time is a single integer: `month_id = (year - 1980) * 12 + month`. The helpers in
`views_crafdapi.time` convert both ways and build ranges.

In [4]:
print("Dec 2024  ->", date_to_month_id(2024, 12))
print("month 540 ->", month_id_to_date(540))
print("Q4 2024   ->", month_id_range(2024, 10, 2024, 12))

Dec 2024  -> 540
month 540 -> 2024-12
Q4 2024   -> [538, 539, 540]


## 2.5 · Global coverage — historical & forecast

The API serves a **global** land grid (the `land_gaul` scope) — every land cell worldwide, not
just Africa — for both historical and forecast. A quick check on two non-African countries
confirms both are live there. Note the feature vocabulary differs: historical exposes the
observed `lr_ged_sb`; forecast exposes the posterior **`pred_lr_ged_sb`**.

In [5]:
# Non-African countries confirm the grid is global for BOTH historical and forecast.
sample = ["COL", "IDN"]  # Colombia, Indonesia
h = fetch_warm(lambda: client.fetch_subset("country", [HIST_MONTH], sample, data_type="historical"))
f = fetch_warm(lambda: client.fetch_subset("country", [FORECAST_MONTH], sample, data_type="forecast",
                                           features=["pred_lr_ged_sb"]))
print(f"historical: {len(h):,} cells across {h['country_iso_a3'].nunique()} non-African countries")
print(f"forecast  : {len(f):,} cells across {f['country_iso_a3'].nunique()} non-African countries")

historical: 1,425 cells across 2 non-African countries
forecast  : 1,425 cells across 2 non-African countries


## 3 · Historical data

`client.fetch_subset(level, time_ids, entity_ids, ...)` returns a tidy `DataFrame`.
`level` is one of `pg`, `country`, `gaul0`, `gaul1`, `gaul2`; entities are PRIO-GRID ints,
ISO3 strings, or GAUL codes depending on the level.

In [6]:
df_hist = client.fetch_subset(
    "country", time_ids=[date_to_month_id(2024, 12)], entity_ids="NGA",
    data_type="historical",
)
print("shape:", df_hist.shape)
df_hist.head()

shape: (310, 14)


,month_id,priogrid_id,lr_ged_sb,lr_ged_ns,lr_ged_os,pg_xcoord,pg_ycoord,country_iso_a3,admin1_gaul1_code,admin1_gaul1_name,admin1_gaul0_code,admin1_gaul0_name,admin2_gaul2_code,admin2_gaul2_name
0,540,135732,0.0,0.0,0.0,5.75,4.25,NGA,1522.0,Bayelsa,151.0,Nigeria,104747.0,Southern Ijaw
1,540,135733,0.0,0.0,0.0,6.25,4.25,NGA,1522.0,Bayelsa,151.0,Nigeria,104741.0,Brass
2,540,135734,0.0,0.0,0.0,6.75,4.25,NGA,1549.0,Rivers,151.0,Nigeria,105315.0,Akuku Toru
3,540,135735,0.0,0.0,0.0,7.25,4.25,NGA,1549.0,Rivers,151.0,Nigeria,105318.0,Bonny
4,540,135736,0.0,0.0,0.0,7.75,4.25,NGA,1549.0,Rivers,151.0,Nigeria,105331.0,Opobo/Nkoro


### 3.1 Levels and features

In [7]:
# A few PRIO-GRID cells, only the state-based feature, three months.
client.fetch_subset(
    "pg",
    time_ids=month_id_range(2024, 10, 2024, 12),
    entity_ids=[174669, 176830],
    data_type="historical",
    features=["lr_ged_sb"],
).head()

,month_id,priogrid_id,lr_ged_sb,pg_xcoord,pg_ycoord,country_iso_a3,admin1_gaul1_code,admin1_gaul1_name,admin1_gaul0_code,admin1_gaul0_name,admin2_gaul2_code,admin2_gaul2_name
0,538,174669,630.0,34.25,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available
1,538,176830,4.0,34.75,32.75,ISR,2700.0,Haifa,244.0,Israel,127043.0,Administrative Unit Not Available
2,539,174669,516.0,34.25,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available
3,539,176830,0.0,34.75,32.75,ISR,2700.0,Haifa,244.0,Israel,127043.0,Administrative Unit Not Available
4,540,174669,547.0,34.25,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available


## 4 · Forecast data — posteriors

Same call with `data_type="forecast"`. The difference: each feature value is an **array of
posterior draws** (the forecast's uncertainty), not a single observed number.

In [8]:
df_fc = client.fetch_subset(
    "pg", time_ids=[FORECAST_MONTH], entity_ids=[174669, 176830, 176831],
    data_type="forecast",
)
print("shape:", df_fc.shape)
df_fc.head()

shape: (3, 14)


,month_id,priogrid_id,pred_lr_ged_sb,pred_lr_ged_ns,pred_lr_ged_os,pg_xcoord,pg_ycoord,country_iso_a3,admin1_gaul1_code,admin1_gaul1_name,admin1_gaul0_code,admin1_gaul0_name,admin2_gaul2_code,admin2_gaul2_name
0,559,174669,"[695.0, 630.0, 1516.0, 84.0, 0.0, 664.0, 75.0,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 519.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 519...",34.25,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available
1,559,176830,"[0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",34.75,32.75,ISR,2700.0,Haifa,244.0,Israel,127043.0,Administrative Unit Not Available
2,559,176831,"[5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 1.0, ...","[0.0, 2.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...",35.25,32.75,ISR,2702.0,Northern,244.0,Israel,127045.0,Administrative Unit Not Available


### 4.1 Posterior samples (`sample_idx`) — raw HTTP

To shrink the payload you can fetch only a few draws per cell. The client doesn't expose
`sample_idx`, so this is a raw request — and a look at the actual endpoint contract.

In [9]:
resp = requests.get(
    f"{BASE_URL}/pg/data/forecast/subset",
    headers=HEADERS,
    params={
        "time_ids": str(FORECAST_MONTH),
        "entity_ids": "174669",
        "sample_idx": "0,1,2,3,4",   # first 5 draws only
        "with_metadata": False,
    },
)
payload = resp.json() if resp.ok else {}
rows = payload.get("data", {}).get("dataframe", [])
rows[0] if rows else f"No rows (HTTP {resp.status_code}) — check the month is in the forecast window (2026-07 … 2029-06) and the cell is on land."

{'month_id': 559,
 'priogrid_id': 174669,
 'pred_lr_ged_sb': [695.0, 630.0, 1516.0, 84.0, 0.0],
 'pred_lr_ged_ns': [0.0, 0.0, 0.0, 0.0, 0.0],
 'pred_lr_ged_os': [0.0, 519.0, 0.0, 0.0, 0.0]}

## 5 · Uncertainty — HDI and MAP

The analysis endpoints collapse each posterior to a **MAP** point estimate and
**Highest-Density Intervals** at the signed-off **50 / 90 / 95 %** credibility levels, per
series (`sb`/`ns`/`os`). Raw requests against `/{level}/analysis/{data_type}/hdi-map`.

In [10]:
resp = requests.get(
    f"{BASE_URL}/pg/analysis/forecast/hdi-map",
    headers=HEADERS,
    params={
        "alpha": 0.9,
        "time_ids": str(FORECAST_MONTH),
        "entity_ids": "174669,174670,174671",
        "with_metadata": True,
    },
)
payload = resp.json() if resp.ok else {}
rows = payload.get("data", {}).get("hdi_map", [])
pd.DataFrame.from_dict(rows).head() if rows else f"No HDI-MAP rows (HTTP {resp.status_code}) — check the month is in the forecast window (2026-07 … 2029-06)."

,month_id,priogrid_id,sb_map,sb_hdi50_lower,sb_hdi50_upper,sb_hdi90_lower,sb_hdi90_upper,sb_hdi95_lower,sb_hdi95_upper,sb_severe_scenario,...,os_p_gt1000,pg_xcoord,pg_ycoord,country_iso_a3,admin1_gaul1_code,admin1_gaul1_name,admin1_gaul0_code,admin1_gaul0_name,admin2_gaul2_code,admin2_gaul2_name
0,559,174669,704.0,75.0,704.0,0.0,1516.0,0.0,1516.0,1516.0,...,0.0,34.25,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available
1,559,174670,0.0,0.0,0.0,0.0,2.0,0.0,2.0,2.0,...,0.0,34.75,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available
2,559,174671,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,35.25,31.25,ISR,2703.0,Southern District,244.0,Israel,127046.0,Administrative Unit Not Available


### 5.1 Credibility levels — wider interval, more confidence

The API serves all three signed-off HDIs — **50 / 90 / 95 %** — in one call. A higher
credibility level gives a wider interval.

In [11]:
# One call returns the 50/90/95% HDIs for the state-based series (sb); higher level -> wider.
resp = requests.get(
    f"{BASE_URL}/pg/analysis/forecast/hdi-map",
    headers=HEADERS,
    params={"time_ids": str(FORECAST_MONTH), "entity_ids": "174669", "with_metadata": False},
)
rows = resp.json().get("data", {}).get("hdi_map", []) if resp.ok else []
if not rows:
    print(f"  (no data, HTTP {resp.status_code}) — check the month is in the forecast window (2026-07 … 2029-06).")
else:
    row = rows[0]
    print(f"  MAP (state-based): {row['sb_map']:.3f}")
    for lvl in (50, 90, 95):
        lo, hi = row[f"sb_hdi{lvl}_lower"], row[f"sb_hdi{lvl}_upper"]
        print(f"  {lvl}% HDI: [{lo:.3f}, {hi:.3f}]  width={hi - lo:.3f}")

  MAP (state-based): 704.000
  50% HDI: [75.000, 704.000]  width=629.000
  90% HDI: [0.000, 1516.000]  width=1516.000
  95% HDI: [0.000, 1516.000]  width=1516.000


### 5.2 Aggregated to a country

With `aggregate=True` the service sums the **aligned posterior draws** across the country's
cells *before* collapsing — so the country's interval reflects joint uncertainty, not a sum
of per-cell intervals. The forecast is global, so this works for any country — here
Nigeria, Somalia, and Colombia together.

In [12]:
resp = requests.get(
    f"{BASE_URL}/country/analysis/forecast/hdi-map",
    headers=HEADERS,
    params={
        "alpha": 0.9,
        "time_ids": ",".join(map(str, month_id_range(2026, 7, 2026, 9))),
        "entity_ids": "NGA,SOM,COL",  # Nigeria, Somalia, Colombia — the forecast is global
        "aggregate": True,
        "enforce_non_negative": True,
    },
)
payload = resp.json() if resp.ok else {}
rows = payload.get("data", {}).get("hdi_map", [])
pd.DataFrame.from_dict(rows) if rows else f"No rows (HTTP {resp.status_code}) — check the months are in the forecast window (2026-07 … 2029-06)."

,month_id,country_iso_a3,sb_map,sb_hdi50_lower,sb_hdi50_upper,sb_hdi90_lower,sb_hdi90_upper,sb_hdi95_lower,sb_hdi95_upper,sb_severe_scenario,...,os_hdi50_upper,os_hdi90_lower,os_hdi90_upper,os_hdi95_lower,os_hdi95_upper,os_severe_scenario,os_bimodality_flag,os_p_gt25,os_p_gt100,os_p_gt1000
0,559,COL,19.0,14.0,19.0,11.0,34.0,11.0,39.0,39.0,...,35.0,19.0,46.0,13.0,46.0,46.0,1.0,0.5625,0.0000,0.0
1,559,NGA,89.0,87.0,138.0,69.0,295.0,69.0,304.0,304.0,...,86.0,4.0,128.0,4.0,142.0,142.0,1.0,0.8750,0.1875,0.0
2,559,SOM,152.0,120.0,186.0,69.0,281.0,69.0,366.0,366.0,...,1.0,0.0,10.0,0.0,11.0,11.0,0.0,0.0000,0.0000,0.0
3,560,COL,19.0,11.0,21.0,7.0,44.0,7.0,49.0,49.0,...,37.0,15.0,46.0,15.0,47.0,47.0,1.0,0.6875,0.0000,0.0
4,560,NGA,100.0,68.0,164.0,58.0,351.0,58.0,381.0,381.0,...,78.0,16.0,120.0,16.0,172.0,172.0,1.0,0.8125,0.2500,0.0
5,560,SOM,153.0,82.0,178.0,82.0,356.0,82.0,482.0,482.0,...,5.0,0.0,15.0,0.0,18.0,18.0,0.0,0.0000,0.0000,0.0
6,561,COL,12.0,10.0,18.0,7.0,37.0,7.0,45.0,45.0,...,41.0,21.0,47.0,18.0,47.0,47.0,1.0,0.5625,0.0000,0.0
7,561,NGA,142.0,101.0,252.0,55.0,284.0,33.0,284.0,284.0,...,130.0,19.0,130.0,19.0,191.0,191.0,1.0,0.9375,0.4375,0.0
8,561,SOM,172.0,137.0,251.0,114.0,307.0,114.0,324.0,324.0,...,2.0,0.0,11.0,0.0,21.0,21.0,0.0,0.0000,0.0000,0.0


## 6 · Endpoint reference

| Purpose | Method & path | Key params |
|---|---|---|
| Health | `GET /health` | — |
| Data subset | `GET /{level}/data/{type}/subset` | `time_ids`, `entity_ids`, `features`, `sample_idx`, `with_metadata` |
| HDI / MAP | `GET /{level}/analysis/{type}/hdi-map` | `alpha`, `time_ids`, `entity_ids`, `aggregate`, `enforce_non_negative` |

`{level}` ∈ `pg, country, gaul0, gaul1, gaul2`; `{type}` ∈ `historical, forecast`.
Every request sends the header `X-API-Key: <your key>`. Interactive docs: `/docs` (Swagger),
`/redoc`.

## Next steps

- **`02_visualization.ipynb`** — turn these subsets into maps and multi-panel comparisons.
- **`03_offline_demo.ipynb`** — the same analytics on synthetic data, **no key required**.
- Full reference: [`docs/api/README.md`](../docs/api/README.md); auth: ADR-027; data source: ADR-028.